# 34 Historical NDJF Catalog: complete seasons 1940/41–2025/26

This is a **separate, checkpointed historical catalog**. It never overwrites the established 2000–2025 catalog or its monthly checkpoints. It reuses compatible existing monthly D12 files read-only, calculates only missing historical months from ERA5/ARCO, then recomputes all monthly mean/std thresholds from the complete historical record. See `docs/historical-catalog-extension.md` for the method and interpretation.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = Path('/content/JPCZcatalog')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/JPCZcatalog_outputs')

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('Google Drive is already mounted.')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

# Explicitly add the cloned source tree as a failsafe for fresh Colab runtimes.
src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.chdir(REPO_DIR)
import jpcz_catalog
print('JPCZ package import verified from:', jpcz_catalog.__file__)
print('Working directory:', Path.cwd())
print('Drive root:', DRIVE_OUTPUT_DIR)

In [ ]:
from pathlib import Path

# Complete seasons: Nov–Dec 1940 through Jan–Feb 2026.
FIRST_SEASON_YEAR = 1940
LAST_SEASON_YEAR = 2025
CATALOG_MONTHS = (11, 12, 1, 2)
MERGE_GAP_HOURS = 12

# Conservative Colab-memory settings.  Each divergence field is reduced to one
# polygon scalar immediately; only one ERA5 analysis hour is loaded at a time.
CHUNK_HOURS = 1
PROGRESS_EVERY_HOURS = 24
MAX_NEW_MONTHS_PER_RUN = 2  # Rerun the monthly-build cell for the next safe batch.

# This versioned name prevents any overwrite of the established catalog.
RUN_ID = 'ndjf_complete_seasons_1940_1941_to_2025_2026'
HISTORICAL_RUN_DIR = DRIVE_OUTPUT_DIR / 'historical_catalogs' / RUN_ID
HISTORICAL_MONTHLY_DIR = HISTORICAL_RUN_DIR / 'monthly_D12_checkpoints'
INVENTORY_PATH = HISTORICAL_RUN_DIR / 'monthly_inventory.csv'
MANIFEST_PATH = HISTORICAL_RUN_DIR / 'run_manifest.txt'

# Existing catalog inputs: read only.
LEGACY_MONTHLY_DIR = DRIVE_OUTPUT_DIR / 'ndjf_monthly'
LEGACY_DECEMBER_DIR = DRIVE_OUTPUT_DIR / 'december_yearly'
ESTABLISHED_MERGED_PATH = DRIVE_OUTPUT_DIR / 'jpcz_catalog_ndjf_merged_12h.csv'

# Toggle one expensive stage at a time.
RUN_MONTHLY_ERA5_BUILD = True
RUN_ENVIRONMENT_CLASSIFICATION = True

for directory in [HISTORICAL_RUN_DIR, HISTORICAL_MONTHLY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f'Historical complete seasons: {FIRST_SEASON_YEAR}/{FIRST_SEASON_YEAR + 1} to {LAST_SEASON_YEAR}/{LAST_SEASON_YEAR + 1}')
print('Monthly build batch limit:', MAX_NEW_MONTHS_PER_RUN)
print('Historical output folder:', HISTORICAL_RUN_DIR)
print('Existing catalog remains read-only at:', DRIVE_OUTPUT_DIR)

In [ ]:
import pandas as pd

from jpcz_catalog.historical_catalog import (
    complete_ndjf_months,
    initialize_historical_inventory,
    write_inventory,
)

season_months = complete_ndjf_months(FIRST_SEASON_YEAR, LAST_SEASON_YEAR)
inventory_rows = initialize_historical_inventory(
    season_months,
    historical_monthly_dir=HISTORICAL_MONTHLY_DIR,
    legacy_monthly_dir=LEGACY_MONTHLY_DIR,
    legacy_december_dir=LEGACY_DECEMBER_DIR,
)
inventory = write_inventory(inventory_rows, INVENTORY_PATH)

manifest = f'''Historical Shinoda-style ERA5 catalog
run_id: {RUN_ID}
complete_seasons: {FIRST_SEASON_YEAR}/{FIRST_SEASON_YEAR + 1} through {LAST_SEASON_YEAR}/{LAST_SEASON_YEAR + 1}
months: November, December, January, February
D12 convention: trailing 12-hour mean, min_periods=12
threshold: full-record monthly mean(D12) - 2 * std(D12)
merge: <= {MERGE_GAP_HOURS} hour quiet gaps
legacy checkpoints: read-only; no established output is overwritten
'''
MANIFEST_PATH.write_text(manifest)

print('Monthly inventory:', INVENTORY_PATH)
display(inventory.groupby(['status', 'source']).size().rename('calendar_months').reset_index())
print(f"Total complete-season months: {len(inventory)}; pending ERA5 calculations: {(inventory.status == 'pending').sum()}")

## Build only missing monthly D12 checkpoints

Each new calendar month is calculated from hourly ERA5 925-hPa winds in 48-hour chunks. Its final D12 series and the inventory are written to Drive before continuing. If Colab disconnects, rerun this cell: completed historical files and compatible 2000–2025 files are skipped automatically.

In [ ]:
import gc
import xarray as xr

from jpcz_catalog.config import JPCZ_POLYGON_VERTICES
from jpcz_catalog.detect import prepare_detection_geometry
from jpcz_catalog.era5 import open_arco_era5, open_month
from jpcz_catalog.historical_catalog import (
    checkpoint_path_for_month,
    compute_polygon_mean_divergence_series_chunked,
)

if RUN_MONTHLY_ERA5_BUILD:
    inventory = pd.read_csv(INVENTORY_PATH)
    pending = inventory.loc[inventory.status == 'pending'].copy().head(MAX_NEW_MONTHS_PER_RUN)
    total_pending = int((inventory.status == 'pending').sum())
    print('Historical D12 inventory:', int(inventory.status.eq('complete').sum()), 'complete;', total_pending, 'pending calendar months.')

    if not pending.empty:
        # Match Notebook 4's low-overhead ARCO opening.  Avoid Dask chunk graphs
        # and load only the one selected ERA5 hour inside the detector helper.
        ds = open_arco_era5()
        print('ARCO time coverage:', ds.attrs.get('valid_time_start'), 'to', ds.attrs.get('valid_time_stop'))
        first = pending.iloc[0]
        geometry_ds = open_month(ds, int(first.year), int(first.month))
        geometry = prepare_detection_geometry(geometry_ds.longitude, geometry_ds.latitude, JPCZ_POLYGON_VERTICES)

        for number, row in enumerate(pending.itertuples(index=False), start=1):
            year, month = int(row.year), int(row.month)
            checkpoint_path = checkpoint_path_for_month(HISTORICAL_MONTHLY_DIR, row)
            print(f'[{number}/{len(pending)} this run] {year}-{month:02d}: selecting ERA5 winds from ARCO...')
            month_ds = open_month(ds, year, month)
            if int(month_ds.sizes.get('time', 0)) < 12:
                raise RuntimeError(f'{year}-{month:02d}: ARCO returned fewer than 12 hourly fields; checkpoint was not written.')
            hourly, D12 = compute_polygon_mean_divergence_series_chunked(
                month_ds, geometry=geometry, chunk_hours=CHUNK_HOURS,
                progress_every_hours=PROGRESS_EVERY_HOURS, progress_label=f'{year}-{month:02d}'
            )
            D_valid = D12.dropna('time')
            D_valid.attrs.update({
                'historical_run_id': RUN_ID,
                'source': 'ERA5 ARCO hourly 925-hPa u and v',
                'processing': 'Shinoda polygon; cosine-latitude weights; trailing 12-hour D12',
            })
            temporary_path = checkpoint_path.with_name(f'.{checkpoint_path.stem}.tmp.nc')
            D_valid.to_netcdf(temporary_path)
            temporary_path.replace(checkpoint_path)
            print(f'{year}-{month:02d}: checkpoint saved to Drive ({D_valid.sizes["time"]} valid D12 hours).')

            # Rebuild and persist the inventory after every month, so reruns resume safely.
            inventory_rows = initialize_historical_inventory(
                season_months, historical_monthly_dir=HISTORICAL_MONTHLY_DIR,
                legacy_monthly_dir=LEGACY_MONTHLY_DIR, legacy_december_dir=LEGACY_DECEMBER_DIR,
            )
            inventory = write_inventory(inventory_rows, INVENTORY_PATH)
            complete_count = int(inventory.status.eq('complete').sum())
            print(f'Progress checkpoint: {complete_count}/{len(inventory)} complete; {len(inventory) - complete_count} pending.')
            del month_ds, hourly, D12, D_valid
            gc.collect()

        remaining = int((inventory.status == 'pending').sum())
        if remaining:
            print(f'Batch complete. {remaining} calendar months remain; rerun this exact cell for the next {MAX_NEW_MONTHS_PER_RUN}.')
    else:
        print('Nothing to calculate: all monthly D12 checkpoints are available.')
else:
    print('RUN_MONTHLY_ERA5_BUILD is False; no ARCO data were requested.')

## Recalculate the full-record thresholds and raw events

This cell does not reuse the 2000–2025 thresholds. It pools every completed monthly D12 series in the historical inventory, calculates a fresh mean and standard deviation for each calendar month, and rebuilds raw threshold-hit fragments.

In [ ]:
from jpcz_catalog.historical_catalog import build_events_from_monthly_checkpoints, write_dataframe_atomic

inventory = pd.read_csv(INVENTORY_PATH)
if (inventory.status != 'complete').any():
    raise RuntimeError('The historical monthly inventory is incomplete. Finish or resume the preceding ERA5 build cell first.')

historical_raw_events, historical_thresholds = build_events_from_monthly_checkpoints(
    inventory, required_months=CATALOG_MONTHS
)
RAW_EVENTS_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_raw_detector.csv'
THRESHOLDS_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_monthly_thresholds.csv'
write_dataframe_atomic(historical_raw_events, RAW_EVENTS_PATH)
write_dataframe_atomic(historical_thresholds, THRESHOLDS_PATH)

print(f'Full-record thresholds saved: {THRESHOLDS_PATH}')
print(f'Raw detector catalog saved: {RAW_EVENTS_PATH}')
print('Raw historical events:', len(historical_raw_events))
display(historical_thresholds)

## Add the Shinoda-style environment classifications

This is post-detection only: it cannot create or remove raw events. The cell checkpoints each event’s SLP-difference and vorticity metrics to the historical Drive folder, then applies the full-record, season-aware SLP climatology and the Type 1B vorticity split: mean vorticity plus one standard deviation.

In [ ]:
import numpy as np

from jpcz_catalog.classify import (
    annotate_events_with_environment,
    compute_slp_index_for_month_records,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.historical_catalog import write_dataframe_atomic

HISTORICAL_CLASSIFIED_RAW_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_raw_classified.csv'
HISTORICAL_SLP_THRESHOLDS_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_slp_climatology.csv'
ENVIRONMENT_CHECKPOINT_PATH = HISTORICAL_RUN_DIR / 'event_environment_checkpoint.csv'

if RUN_ENVIRONMENT_CLASSIFICATION:
    if 'ds' not in globals():
        ds = open_arco_era5(chunks={})
    historical_raw_events = pd.read_csv(RAW_EVENTS_PATH, parse_dates=['event_start', 'event_end', 'event_peak'])
    print(f'Historical classification: {len(historical_raw_events)} raw events.')
    historical_catalog = annotate_events_with_environment(
        ds, historical_raw_events, checkpoint_path=ENVIRONMENT_CHECKPOINT_PATH,
        checkpoint_every=1, progress_label='Historical event environment',
    )

    inventory = pd.read_csv(INVENTORY_PATH)
    slp_rows, slp_means, slp_stds = [], {}, {}
    for month in CATALOG_MONTHS:
        pairs = list(inventory.loc[inventory.month == month, ['year', 'month']].itertuples(index=False, name=None))
        _, mean_value, std_value = compute_slp_index_for_month_records(
            ds, pairs, progress_every=20, progress_label=f'Historical SLP month {month:02d}'
        )
        slp_means[month] = mean_value
        slp_stds[month] = std_value
        slp_rows.append({'month': month, 'slp_climatology_mean_hpa': mean_value, 'slp_climatology_std_hpa': std_value})

    historical_catalog['slp_climatology_mean_hpa'] = historical_catalog['month'].map(slp_means)
    historical_catalog['slp_climatology_std_hpa'] = historical_catalog['month'].map(slp_stds)
    historical_catalog['monsoon_type'] = np.where(
        historical_catalog['slp_diff_hpa'] > historical_catalog['slp_climatology_mean_hpa'],
        'Type 1 strong-monsoon', 'Type 2 weak-monsoon',
    )
    type1 = historical_catalog.monsoon_type == 'Type 1 strong-monsoon'
    zeta_split = historical_catalog.loc[type1, 'zeta_box_mean_s-1'].mean() + historical_catalog.loc[type1, 'zeta_box_mean_s-1'].std()
    historical_catalog['shinoda_class'] = 'Type 2 weak-monsoon'
    historical_catalog.loc[type1 & (historical_catalog['zeta_box_mean_s-1'] > zeta_split), 'shinoda_class'] = 'Type 1B higher-vorticity'
    historical_catalog.loc[type1 & (historical_catalog['zeta_box_mean_s-1'] <= zeta_split), 'shinoda_class'] = 'Type 1A lower-vorticity'

    write_dataframe_atomic(pd.DataFrame(slp_rows).sort_values('month'), HISTORICAL_SLP_THRESHOLDS_PATH)
    write_dataframe_atomic(historical_catalog, HISTORICAL_CLASSIFIED_RAW_PATH)
    print('Classified raw catalog saved:', HISTORICAL_CLASSIFIED_RAW_PATH)
    print('Historical Type 1B vorticity threshold (mean + SD):', zeta_split, 's^-1')
else:
    print('RUN_ENVIRONMENT_CLASSIFICATION is False; this cell made no changes.')

## Build the final merged catalog and compare it with the established catalog

The same 12-hour-or-less quiet-gap merge rule is applied. The comparison is by inclusive temporal overlap, not exact peak timestamp: a changed climatological threshold can shorten or extend the same physical episode. The key output lists every established merged episode that is not flagged by the full historical baseline.

In [ ]:
from jpcz_catalog.detect import merge_gap_sensitivity, merge_nearby_events
from jpcz_catalog.historical_catalog import compare_merged_catalogs, write_dataframe_atomic

if not HISTORICAL_CLASSIFIED_RAW_PATH.exists():
    raise FileNotFoundError('Run the historical classification cell before building the final merged catalog.')
if not ESTABLISHED_MERGED_PATH.exists():
    raise FileNotFoundError(f'Established merged catalog not found: {ESTABLISHED_MERGED_PATH}')

historical_catalog = pd.read_csv(HISTORICAL_CLASSIFIED_RAW_PATH, parse_dates=['event_start', 'event_end', 'event_peak'])
historical_sensitivity = merge_gap_sensitivity(historical_catalog, gap_hours_values=(6, 12, 24))
historical_merged = merge_nearby_events(historical_catalog, max_gap_hours=MERGE_GAP_HOURS)

HISTORICAL_MERGED_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_merged_12h.csv'
HISTORICAL_SENSITIVITY_PATH = HISTORICAL_RUN_DIR / 'jpcz_catalog_ndjf_1940_2026_gap_merge_sensitivity.csv'
write_dataframe_atomic(historical_merged, HISTORICAL_MERGED_PATH)
write_dataframe_atomic(historical_sensitivity, HISTORICAL_SENSITIVITY_PATH)

established_merged = pd.read_csv(ESTABLISHED_MERGED_PATH, parse_dates=['event_start', 'event_end', 'event_peak'])
comparison, comparison_summary = compare_merged_catalogs(established_merged, historical_merged)
COMPARISON_PATH = HISTORICAL_RUN_DIR / 'comparison_established_2000_2025_vs_historical_1940_2026_merged.csv'
COMPARISON_SUMMARY_PATH = HISTORICAL_RUN_DIR / 'comparison_established_2000_2025_vs_historical_1940_2026_summary.csv'
write_dataframe_atomic(comparison, COMPARISON_PATH)
write_dataframe_atomic(comparison_summary, COMPARISON_SUMMARY_PATH)

print('Historical final merged catalog:', HISTORICAL_MERGED_PATH)
print('Comparison table:', COMPARISON_PATH)
display(historical_sensitivity)
display(comparison_summary)

not_flagged = comparison.loc[comparison.comparison_status == 'established_only_not_flagged_historical'].copy()
print(f'Established final merged episodes not flagged by the historical baseline: {len(not_flagged)}')
display(not_flagged.sort_values('established_peak'))